# 02. Tier 2 Semantic Layer: Stateful Sessionization & Transition Tracking

### Architectural Purpose 
In `01_bronze_ingestion`, we discovered that raw event logs contain **42.45M stateless rows** plagued by funnel inversions (direct 1-click purchases bypassing `cart` events) and ~32% taxonomy nulls. Pointing an LLM agent directly at these raw rows forces it to dynamically reconstruct stateful window logic on every query—leading to high token burn and silent SQL hallucinations.

In this notebook, we engineer our **Tier 2 Semantic Layer (`workspace.default.silver_sessions`)** in PySpark:  
1. **Audit Raw Session Defects:** Quantify multi-user collisions, >24-hour zombie sessions, and automated bot bursts in the raw `user_session` UUIDs.
2. **30-Minute Inactivity Re-Sessionization:** Apply `Window` + `LAG()` functions over `(user_id, user_session)` to split multi-day sessions into true behavioral episodes (`clean_session_id`).
3. **Within-Session Transition Detection:** Capture step-by-step `is_brand_switch` and `is_price_step_down` signals—which serve both as semantic features here and as explicit relationship edges in our **Tier 3 Ontology** (`03_ontology_and_sampling`).
4. **Token-Compact Sequence Serialization:** Compress up to 15 chronological events per session into a single array of compact strings (`action_sequence`) for downstream LLM semantic tagging. 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Tier 1 Bronze table (42,448,764 raw events)
bronze = spark.table("workspace.default.bronze_events")

# Profile raw `user_session` UUIDs before applying any cleaning rules.
# These metrics prove why naive GROUP BY user_session queries distort conversion & duration KPIs:
#   - Multi-user sessions: shared device or UUID reuse across distinct user_ids
#   - >24h sessions: unclosed browser tabs spanning days or weeks
#   - Bot bursts (>200 events): automated scrapers skewing view-to-purchase ratios
session_anomalies = (
    bronze.filter(F.col("user_session").isNotNull())
    .groupBy("user_session")
    .agg(
        F.count("*").alias("events_in_session"),
        F.countDistinct("user_id").alias("users_per_session"),
        (F.unix_timestamp(F.max("event_ts")) - F.unix_timestamp(F.min("event_ts"))).alias("duration_sec")
    )
)

anomaly_summary = session_anomalies.agg(
    F.count("*").alias("total_raw_sessions"),
    F.sum((F.col("users_per_session") > 1).cast("int")).alias("multi_user_sessions"),
    F.sum((F.col("duration_sec") > 86400).cast("int")).alias("sessions_over_24h"),
    F.sum((F.col("events_in_session") > 200).cast("int")).alias("bot_burst_sessions_gt_200")
)

# .show() persists plain-text output for GitHub .ipynb rendering; display() renders Databricks UI
anomaly_summary.show(truncate=False)
display(anomaly_summary)


In [0]:
# Step 3A: Partition by (user_id, user_session) to prevent cross-user collisions on shared UUIDs
w_user_sess = Window.partitionBy("user_id", "user_session").orderBy("event_ts")

events_enriched = (
    bronze
    # Drop records missing core entity identifiers required for stateful session tracking
    .filter(F.col("user_session").isNotNull() & F.col("user_id").isNotNull())
    # Calculate time elapsed since the user's previous event in the same raw session
    .withColumn("prev_ts", F.lag("event_ts").over(w_user_sess))
    .withColumn("gap_sec", F.unix_timestamp("event_ts") - F.unix_timestamp("prev_ts"))
    # Start a new behavioral episode whenever inactivity exceeds 30 minutes (1,800 seconds)
    .withColumn(
        "new_episode_flag",
        F.when((F.col("gap_sec").isNull()) | (F.col("gap_sec") > 1800), 1).otherwise(0)
    )
    # Cumulative sum of episode boundaries assigns a deterministic episode index (1, 2, 3...)
    .withColumn("episode_num", F.sum("new_episode_flag").over(w_user_sess))
    .withColumn("clean_session_id", F.concat_ws("_", F.col("user_session"), F.col("episode_num")))
    # Normalize hierarchical taxonomy & impute nulls so LLM GROUP BY queries never drop rows
    .withColumn("top_category", F.coalesce(F.split(F.col("category_code"), r"\.").getItem(0), F.lit("uncategorized")))
    .withColumn("clean_brand", F.coalesce(F.col("brand"), F.lit("unknown")))
)

# Step 3B: Secondary window within each clean 30-min episode to capture step-by-step product comparisons
w_clean_sess = Window.partitionBy("clean_session_id").orderBy("event_ts")

events_with_transitions = (
    events_enriched
    .withColumn("prev_brand", F.lag("clean_brand").over(w_clean_sess))
    .withColumn("prev_price", F.lag("price").over(w_clean_sess))
    .withColumn("prev_category", F.lag("top_category").over(w_clean_sess))
    # Flag explicit brand switching within the same product category (feeds Tier 3 Ontology edges)
    .withColumn(
        "is_brand_switch",
        F.when(
            (F.col("prev_brand").isNotNull()) &
            (F.col("prev_brand") != "unknown") &
            (F.col("clean_brand") != "unknown") &
            (F.col("prev_brand") != F.col("clean_brand")) &
            (F.col("prev_category") == F.col("top_category")),
            1
        ).otherwise(0)
    )
    # Flag price-sensitive step-downs within the same product category (feeds Tier 3 Ontology edges)
    .withColumn(
        "is_price_step_down",
        F.when(
            (F.col("prev_price").isNotNull()) &
            (F.col("price") < F.col("prev_price")) &
            (F.col("prev_category") == F.col("top_category")),
            1
        ).otherwise(0)
    )
)

print("Window enrichment & transition logic compiled successfully.")


In [0]:
# Aggregate 42.45M stateless events into one stateful row per clean_session_id.
silver_sessions = (
    events_with_transitions
    .groupBy("clean_session_id", "user_id", "user_session")
    .agg(
        # Session temporal bounds & total active duration in seconds
        F.min("event_ts").alias("session_start_ts"),
        F.max("event_ts").alias("session_end_ts"),
        (F.unix_timestamp(F.max("event_ts")) - F.unix_timestamp(F.min("event_ts"))).alias("duration_sec"),
        # Funnel interaction counts
        F.count("*").alias("total_events"),
        F.sum((F.col("event_type") == "view").cast("int")).alias("view_count"),
        F.sum((F.col("event_type") == "cart").cast("int")).alias("cart_count"),
        F.sum((F.col("event_type") == "purchase").cast("int")).alias("purchase_count"),
        # Behavioral transition counters
        F.sum("is_brand_switch").alias("brand_switch_count"),
        F.sum("is_price_step_down").alias("price_step_down_count"),
        # Catalog breadth & primary entity attributes
        F.countDistinct("product_id").alias("distinct_products"),
        F.countDistinct("top_category").alias("distinct_categories"),
        F.first("top_category", ignorenulls=True).alias("primary_category"),
        F.first("clean_brand", ignorenulls=True).alias("primary_brand"),
        F.last("clean_brand", ignorenulls=True).alias("final_brand"),
        # Monetary metrics
        F.round(F.avg("price"), 2).alias("avg_price_viewed"),
        F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(0.0)), 2).alias("session_revenue"),
        # Serialize chronological events into compact tokens (e.g., "view:electronics/samsung($250)")
        # capped at 15 events to bound LLM context window consumption.
        F.slice(
            F.transform(
                F.array_sort(F.collect_list(F.struct("event_ts", "event_type", "top_category", "clean_brand", "price"))),
                lambda x: F.concat(
                    x.event_type, F.lit(":"), x.top_category, F.lit("/"), x.clean_brand,
                    F.lit("($"), F.round(x.price, 0).cast("string"), F.lit(")")
                )
            ),
            1, 15
        ).alias("action_sequence")
    )
    # Flag probable automated scrapers/bots (>100 events/session or <0.5s average inter-click interval)
    .withColumn(
        "is_bot_suspect",
        (F.col("total_events") > 100) | ((F.col("total_events") > 10) & ((F.col("duration_sec") / F.col("total_events")) < 0.5))
    )
    # Classify deterministic funnel states—explicitly separating direct 1-click purchases from cart checkouts
    .withColumn(
        "funnel_outcome",
        F.when((F.col("purchase_count") > 0) & (F.col("cart_count") == 0), F.lit("direct_purchase_no_cart"))
         .when(F.col("purchase_count") > 0, F.lit("standard_purchase"))
         .when(F.col("cart_count") > 0, F.lit("cart_abandoned"))
         .when(F.col("view_count") == 1, F.lit("single_event_bounce"))
         .otherwise(F.lit("multi_view_browse"))
    )
)

# Persist compressed Silver table to Unity Catalog
(
    silver_sessions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_sessions")
)

print("Successfully materialized workspace.default.silver_sessions!")


In [0]:
silver = spark.table("workspace.default.silver_sessions")

clean_count = silver.count()
print(f"Total clean stateful sessions: {clean_count:,} (compressed from 42,448,764 raw events)")

# Summarize session distribution, bot traffic, brand switching, and revenue by funnel outcome
funnel_summary = (
    silver.groupBy("funnel_outcome", "is_bot_suspect")
    .agg(
        F.count("*").alias("session_count"),
        F.round(F.avg("total_events"), 1).alias("avg_events"),
        F.round(F.avg("duration_sec"), 1).alias("avg_duration_sec"),
        F.round(F.avg("brand_switch_count"), 2).alias("avg_brand_switches"),
        F.round(F.sum("session_revenue"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("session_count"))
)

funnel_summary.show(truncate=False)
display(funnel_summary)
